# Демо: ИИ-куратор EduPath

Ноутбук показывает тот же пайплайн, что и Streamlit: индексация → RAG → сценарии → ответ со ссылкой на источник.

Данные учебные и вымышленные. Запуск ячеек по порядку.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "edu_curator").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

In [ ]:
from edu_curator.rag.indexer import index_knowledge
from edu_curator.rag.loader import load_knowledge_documents
from edu_curator.rag.retriever import KnowledgeRetriever
from edu_curator.curator.scenarios import classify_scenario
from edu_curator.curator.agent import CuratorAgent
from edu_curator.lms.stub import LMSStub
from edu_curator.models import CuratorRequest, StudentLevel

meta = index_knowledge()
docs = load_knowledge_documents()
print(meta)
print("Документов в базе:", len(docs))

## Поиск с фильтром по метаданным

In [ ]:
retriever = KnowledgeRetriever()
hits = retriever.search("дедлайн домашнего задания pandas", resource_type="faq", k=3)
for h in hits:
    print(f"{h.doc_id:12} {h.module:4} {h.resource_type:10} score={h.score:.3f}  {h.topic}")

## Сценарии и ответ куратора

In [ ]:
questions = [
    "Когда дедлайн первого домашнего задания?",
    "Я ничего не понимаю в корреляции, слишком сложно",
    "Поставь мне зачёт и перенеси дедлайн",
]
for q in questions:
    print(q, "→", classify_scenario(q).value)

print("\n--- LMS факты (фрагмент) ---\n")
print("\n".join(LMSStub().facts_for_prompt().splitlines()[:12]))

In [ ]:
agent = CuratorAgent()
print("LLM:", agent.llm_backend, "| retrieval:", agent.retriever.backend)

response = agent.ask(
    CuratorRequest(
        question="Когда дедлайн первого домашнего задания по pandas?",
        level=StudentLevel.BEGINNER,
        student_id="STU-1001",
        student_name="Анна П.",
    ),
    persist=False,
)
print("Сценарий:", response.scenario.value)
print(response.answer)

In [ ]:
adv = agent.ask(
    CuratorRequest(
        question="Что почитать сверх лекции по ML, чтобы не скучать?",
        level=StudentLevel.ADVANCED,
        student_id="STU-1002",
        student_name="Дмитрий С.",
    ),
    persist=False,
)
print(adv.answer)